# Vector Version-Aware Upsert and Retire Demo (Offline, API-Key-Free)

This notebook implements the **upsert-new-then-retire-old** write pattern proposed in
`../08-vector-index-staleness-and-document-revision-handling.md` (Part 4), against the same
`FakePineconeIndex` stand-in from `02_pinecone_vector_search_demo.ipynb`'s Chapter 2 demo.

The chapter's honest framing applies here too: this is a **proposed design**, not a description of
code that exists. The goal is to show, concretely, why a query against un-versioned vectors can return
stale content, and how `document_version` / `supersedes_id` / `is_current` metadata plus an
`is_current` retrieval filter closes that gap -- the same conceptual fix Chapter 8 describes for
Project Atlas's status.

In [1]:
import numpy as np
import re

np.random.seed(42)
print("Ready.")

Ready.


## 1. Reuse the deterministic fake-embedding function and `FakePineconeIndex`

Identical to `02_pinecone_vector_search_demo.ipynb` -- copied here rather than imported so this
notebook runs standalone. In a real codebase these would live in a shared `utils` module imported by
both notebooks/services.

In [2]:
EMBED_DIM = 32

def fake_embed(text: str) -> list:
    """Deterministic bag-of-words hashing embedding -- a stand-in for a real embedding model call."""
    vec = np.zeros(EMBED_DIM)
    words = re.findall(r"[a-zA-Z0-9\-]+", text.lower())
    for w in words:
        idx = hash(w) % EMBED_DIM
        vec[idx] += 1.0
    norm = np.linalg.norm(vec)
    return (vec / norm if norm > 0 else vec).tolist()


class FakePineconeIndex:
    """In-memory stand-in for pinecone.Index with the same method signatures as
    02_pinecone_vector_search_demo.ipynb's version, plus `.update()` for the metadata-only write the
    retire step in Part 4 of Chapter 8 needs (pinecone.Index.update(id=..., set_metadata=...))."""

    def __init__(self, dimension: int, metric: str = "cosine"):
        self.dimension = dimension
        self.metric = metric
        # namespace -> {id: {"values": [...], "metadata": {...}}}
        self._store: dict[str, dict[str, dict]] = {}

    def upsert(self, vectors: list[dict], namespace: str = "") -> dict:
        ns = self._store.setdefault(namespace, {})
        for v in vectors:
            ns[v["id"]] = {"values": v["values"], "metadata": v.get("metadata", {})}
        return {"upserted_count": len(vectors)}

    def update(self, id: str, set_metadata: dict, namespace: str = "") -> dict:
        """Metadata-only patch -- mirrors pinecone.Index.update(id=..., set_metadata=..., namespace=...).
        Used by the retire step to flip is_current without re-embedding or moving the vector."""
        ns = self._store.get(namespace, {})
        if id not in ns:
            raise KeyError(f"id {id!r} not found in namespace {namespace!r}")
        ns[id]["metadata"].update(set_metadata)
        return {"updated": id}

    def delete(self, ids: list[str], namespace: str = "") -> dict:
        ns = self._store.get(namespace, {})
        for doc_id in ids:
            ns.pop(doc_id, None)
        return {"deleted_count": len(ids)}

    def _matches_filter(self, metadata: dict, filt: dict | None) -> bool:
        if not filt:
            return True
        for key, cond in filt.items():
            val = metadata.get(key)
            if isinstance(cond, dict):
                if "$eq" in cond and val != cond["$eq"]:
                    return False
                if "$in" in cond and val not in cond["$in"]:
                    return False
            else:
                if val != cond:
                    return False
        return True

    def query(self, vector: list, top_k: int = 5, namespace: str = "",
              filter: dict | None = None, include_metadata: bool = False) -> dict:
        ns = self._store.get(namespace, {})
        q = np.array(vector)
        scored = []
        for doc_id, entry in ns.items():
            if not self._matches_filter(entry["metadata"], filter):
                continue
            v = np.array(entry["values"])
            sim = float(np.dot(q, v) / (np.linalg.norm(q) * np.linalg.norm(v) + 1e-9))
            scored.append((doc_id, sim, entry["metadata"]))
        scored.sort(key=lambda t: t[1], reverse=True)
        matches = [
            {"id": doc_id, "score": score, **({"metadata": md} if include_metadata else {})}
            for doc_id, score, md in scored[:top_k]
        ]
        return {"matches": matches, "namespace": namespace}


index = FakePineconeIndex(dimension=EMBED_DIM, metric="cosine")
print("Fake index created, dimension:", index.dimension)

Fake index created, dimension: 32


## 2. Seed the index the way it looks *before* the Part 4 fix -- no version metadata at all

This mirrors Chapter 8 Part 1's status quo: a `project_id` / `doc_type` chunk gets a plain upsert
whenever the status changes, under a stable ID, with no `document_version` / `is_current` fields.
Project Atlas starts "on track for Q3 launch." 

In [3]:
client_id = "eli-lilly"
chunk_id = "atlas-status"

index.upsert(vectors=[{
    "id": chunk_id,
    "values": fake_embed("Project Atlas Japan localization on track for Q3 launch"),
    "metadata": {"client_id": client_id, "project_id": "atlas", "doc_type": "status_update"},
}], namespace=client_id)

print(index.query(vector=fake_embed("What is the status of Project Atlas?"),
                   top_k=3, namespace=client_id, include_metadata=True))

{'matches': [{'id': 'atlas-status', 'score': 0.3418817290370321, 'metadata': {'client_id': 'eli-lilly', 'project_id': 'atlas', 'doc_type': 'status_update'}}], 'namespace': 'eli-lilly'}


## 3. The status changes -- and, pre-fix, the update just overwrites the same ID with no history

Project Atlas is delayed. A realistic sync worker (Chapter 8 Part 1's "upsert-on-change trigger")
re-embeds and re-upserts under the same `chunk_id`. This works fine in the *simple* re-embed case --
but it has no notion of "version," so nothing downstream can tell a change happened, and nothing
prevents a subtler failure mode explored below: a **retrieval path that filters on stale-looking
criteria and misses the update, or a concurrent read racing the write**, returning what looks like a
plausible but outdated answer with no signal anything is wrong.

To make the "stale content returned" failure concrete and reproducible (rather than hand-wavy), the
cells below simulate the *specific* scenario Chapter 8 Part 4 is designed to prevent: two versions of
the same logical chunk momentarily both queryable, with **no `is_current` field to disambiguate which
one is the source of truth**. That is exactly what happens if a naive fix stores the new version under
a *new* ID (e.g. so the old one is kept for audit purposes) without also marking it retired.

In [4]:
# Naive "fix" for wanting an audit trail: store the new version under a new ID, but forget to mark
# the old one as no longer current. This is the bug Chapter 8 Part 4 exists to prevent.
stale_id = "atlas-status"                    # the original, still sitting in the index unmarked
new_id_naive = "atlas-status-v2-naive"

index.upsert(vectors=[{
    "id": new_id_naive,
    "values": fake_embed("Project Atlas Japan localization delayed two weeks legal review"),
    "metadata": {"client_id": client_id, "project_id": "atlas", "doc_type": "status_update"},
}], namespace=client_id)

# Both the stale and the fresh vector are now live, and nothing in the metadata distinguishes them.
before_fix = index.query(
    vector=fake_embed("What is the status of Project Atlas Japan localization?"),
    top_k=3, namespace=client_id, include_metadata=True,
)
for m in before_fix["matches"]:
    print(m["id"], round(m["score"], 4), m["metadata"])

atlas-status-v2-naive 0.6472 {'client_id': 'eli-lilly', 'project_id': 'atlas', 'doc_type': 'status_update'}
atlas-status 0.5025 {'client_id': 'eli-lilly', 'project_id': 'atlas', 'doc_type': 'status_update'}


Notice the top match is not guaranteed to be the fresh one -- both vectors are equally eligible,
and whichever has marginally higher cosine similarity to the query wins, regardless of which one is
actually current. Cell above prints both; **which one ranks first is exactly the kind of thing Chapter
8 Part 3 warns about**: the similarity math is completely valid for both, and nothing about the score
tells you which is stale. Run the cell below to see the "stale wins" failure mode directly, using a
query phrased closer to the original (stale) wording.

In [5]:
stale_leaning_query = index.query(
    vector=fake_embed("Project Atlas Japan localization on track"),
    top_k=1, namespace=client_id, include_metadata=True,
)
top_match = stale_leaning_query["matches"][0]
print("Top match for a stale-leaning query:", top_match["id"])
print("Content this vector represents (by construction):",
      "STALE (on-track)" if top_match["id"] == stale_id else "FRESH (delayed)")
assert top_match["id"] == stale_id, "expected the stale vector to win this query, demonstrating the gap"
print("\nConfirmed: without version metadata, a query can retrieve the stale vector even though a")
print("newer version exists in the same index.")

Top match for a stale-leaning query: atlas-status
Content this vector represents (by construction): STALE (on-track)

Confirmed: without version metadata, a query can retrieve the stale vector even though a
newer version exists in the same index.


## 4. The Part 4 fix: `document_version` / `supersedes_id` / `is_current` metadata, and the
upsert-new-then-retire-old write pattern

This is `upsert_new_version` from `../08-vector-index-staleness-and-document-revision-handling.md`
Part 4, implemented against `FakePineconeIndex` above. Reset to a clean index so the naive-bug vectors
from Section 3 don't leak into this demonstration.

In [6]:
index = FakePineconeIndex(dimension=EMBED_DIM, metric="cosine")

def upsert_new_version(new_content: str, prior_vector_id: str | None, chunk_id: str,
                        current_version: int, metadata_base: dict) -> str:
    """Implements Chapter 8 Part 4's write pattern: write the new version FIRST, retire the old one
    only after the new one is confirmed written. Returns the new vector's id."""
    new_vector_id = f"{chunk_id}-v{current_version + 1}"

    # Step 1: write the new version first.
    index.upsert(vectors=[{
        "id": new_vector_id,
        "values": fake_embed(new_content),
        "metadata": {
            **metadata_base,
            "document_version": current_version + 1,
            "supersedes_id": prior_vector_id,
            "is_current": True,
        },
    }], namespace=metadata_base["client_id"])

    # Step 2: retire the old version only after the new one is confirmed written (soft-delete, keeps
    # the superseded vector queryable-by-explicit-ID for an audit trail, per Part 4's reasoning).
    if prior_vector_id is not None:
        index.update(id=prior_vector_id, set_metadata={"is_current": False},
                     namespace=metadata_base["client_id"])

    return new_vector_id


metadata_base = {"client_id": client_id, "project_id": "atlas", "doc_type": "status_update"}

# v1: initial upsert, no prior version.
v1_id = upsert_new_version(
    new_content="Project Atlas Japan localization on track for Q3 launch",
    prior_vector_id=None, chunk_id="atlas-status", current_version=0,
    metadata_base=metadata_base,
)
print("v1 written as:", v1_id)

v1 written as: atlas-status-v1


In [7]:
# v2: the status changes -- delayed. This is the "content revised" trigger from Part 1.
v2_id = upsert_new_version(
    new_content="Project Atlas Japan localization delayed two weeks legal review",
    prior_vector_id=v1_id, chunk_id="atlas-status", current_version=1,
    metadata_base=metadata_base,
)
print("v2 written as:", v2_id)

# Confirm the write-then-retire ordering worked: v1 still exists (audit trail) but is marked stale.
raw_v1 = index._store[client_id][v1_id]
raw_v2 = index._store[client_id][v2_id]
print("\nv1 metadata after retire:", raw_v1["metadata"])
print("v2 metadata:", raw_v2["metadata"])
assert raw_v1["metadata"]["is_current"] is False
assert raw_v2["metadata"]["is_current"] is True
assert raw_v2["metadata"]["supersedes_id"] == v1_id
print("\nWrite-then-retire ordering confirmed: v1 preserved but marked not-current, v2 is current.")

v2 written as: atlas-status-v2

v1 metadata after retire: {'client_id': 'eli-lilly', 'project_id': 'atlas', 'doc_type': 'status_update', 'document_version': 1, 'supersedes_id': None, 'is_current': False}
v2 metadata: {'client_id': 'eli-lilly', 'project_id': 'atlas', 'doc_type': 'status_update', 'document_version': 2, 'supersedes_id': 'atlas-status-v1', 'is_current': True}

Write-then-retire ordering confirmed: v1 preserved but marked not-current, v2 is current.


## 5. The `is_current` retrieval filter -- Part 4's defense-in-depth query pattern

`retrieve_for_client` from Chapter 8 Part 4, applied here: every retrieval call adds
`is_current: {"$eq": True}` to its filter, on top of `project_id` / `doc_type`. This is the piece
that makes the fix enforceable even if a future retire step is missed or delayed.

In [8]:
def retrieve_for_client(query_text: str, client_id: str, project_id: str | None = None,
                          doc_type: str | None = None, top_k: int = 5):
    filt = {"is_current": {"$eq": True}}
    if project_id:
        filt["project_id"] = {"$eq": project_id}
    if doc_type:
        filt["doc_type"] = {"$eq": doc_type}
    return index.query(
        vector=fake_embed(query_text), top_k=top_k, namespace=client_id,
        filter=filt, include_metadata=True,
    )


# Same stale-leaning query that fooled the naive (unversioned) index in Section 3.
after_fix = retrieve_for_client(
    "Project Atlas Japan localization on track", client_id=client_id, project_id="atlas",
    doc_type="status_update", top_k=3,
)
print("Matches after the fix (is_current filter applied):")
for m in after_fix["matches"]:
    print(" ", m["id"], round(m["score"], 4), m["metadata"])

assert len(after_fix["matches"]) == 1, "only the current version should ever be retrievable"
assert after_fix["matches"][0]["id"] == v2_id, "the fresh (delayed) version must win, not the stale one"
print("\nFix confirmed: the is_current filter excludes v1 entirely, regardless of how closely the")
print("query wording matches the stale version's text -- the fresh version is the only one eligible.")

Matches after the fix (is_current filter applied):
  atlas-status-v2 0.6794 {'client_id': 'eli-lilly', 'project_id': 'atlas', 'doc_type': 'status_update', 'document_version': 2, 'supersedes_id': 'atlas-status-v1', 'is_current': True}

Fix confirmed: the is_current filter excludes v1 entirely, regardless of how closely the
query wording matches the stale version's text -- the fresh version is the only one eligible.


## 6. Closing the deletion gap the same way: a cancelled project reduces to `is_current: False`

Chapter 8 Part 4's closing point: "this project was cancelled" and "this project was revised" both
reduce to the same operation. No separate deletion pipeline is needed -- retiring a cancelled
project's vector is just another `is_current` flip, with no replacement version written.

In [9]:
def retire_without_replacement(prior_vector_id: str, client_id: str) -> None:
    """A project is cancelled/archived -- there is no new version, just a retire. Mirrors the
    delete-on-removal gap Chapter 8 Part 1 names and Part 4 closes."""
    index.update(id=prior_vector_id, set_metadata={"is_current": False}, namespace=client_id)


retire_without_replacement(v2_id, client_id)

after_cancellation = retrieve_for_client(
    "What is the status of Project Atlas?", client_id=client_id, project_id="atlas",
    doc_type="status_update", top_k=3,
)
print("Matches after Project Atlas is cancelled:", after_cancellation["matches"])
assert after_cancellation["matches"] == [], \
    "a cancelled project's vectors must not be retrievable once retired"
print("\nConfirmed: once retired (whether by supersession or cancellation), a vector is excluded from")
print("every is_current-filtered retrieval call -- the client asking about Atlas now correctly gets no")
print("grounding, rather than a confident answer synthesized from dead data.")

Matches after Project Atlas is cancelled: []

Confirmed: once retired (whether by supersession or cancellation), a vector is excluded from
every is_current-filtered retrieval call -- the client asking about Atlas now correctly gets no
grounding, rather than a confident answer synthesized from dead data.


## Takeaways

- **Before the fix**: two versions of the same logical chunk can coexist in the index with nothing to
  distinguish "current" from "stale" -- a query can retrieve either one, and the similarity score gives
  no hint which is which (Section 3).
- **After the fix**: `upsert_new_version` writes the new vector first, then retires the prior one by
  flipping `is_current: False` rather than deleting it -- preserving an audit trail while making the old
  vector permanently excluded from any filtered retrieval call (Section 4-5).
- **The `is_current` filter is defense in depth, not just a write-time convention** -- even a retrieval
  path a future engineer forgets to update stays correct as long as it inherits the shared
  `retrieve_for_client` helper, exactly as Chapter 8 Part 4 argues.
- **Deletion and revision are the same operation** under this design -- both reduce to `is_current:
  False`, which is why Chapter 8 Part 4 closes the Part 1 "no delete-on-removal path" gap without a
  separate deletion pipeline (Section 6).